<a href="https://colab.research.google.com/github/mirdbg/Entrega_RAG/blob/main/Entrega_RAG_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente para el análisis de informes financieros 10-K

**Práctica de LLMs aplicados a Finanzas · MIAX**

## Introducción

Este proyecto desarrolla un **agente basado en LLMs para consultar y analizar
informes financieros 10-K de la SEC**. El objetivo es construir un sistema capaz
de responder preguntas sobre información financiera combinando distintas fuentes
y estrategias de recuperación, seleccionando en cada caso la herramienta más
adecuada.

En lugar de implementar un pipeline RAG clásico, en el que la recuperación de
información se ejecuta siempre antes de generar una respuesta, se utiliza una
**arquitectura agéntica**. En ella, el propio modelo decide qué información
necesita, qué herramienta utilizar, con qué consulta y cuántas veces debe
ejecutarla antes de producir la respuesta final.

## De RAG clásico a Agentic RAG

Un RAG clásico sigue normalmente un flujo fijo:

1. La documentación se divide en fragmentos.
2. Se calculan sus *embeddings*.
3. Los fragmentos se almacenan en un índice vectorial.
4. Ante una pregunta, se recuperan los *k* fragmentos más similares.
5. Esos fragmentos se incorporan al contexto del LLM para generar la respuesta.

Este enfoque, formalizado por Lewis et al. (2020) en
[*Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*](https://arxiv.org/abs/2005.11401),
constituye una solución eficaz para muchas tareas de consulta documental. Sin
embargo, presenta limitaciones relevantes para el análisis de información
financiera.

En particular, algunas preguntas requieren **combinar información procedente de
varios documentos o periodos**, otras necesitan recuperar una **cifra exacta
almacenada en información estructurada**, y otras pueden hacer referencia a
información que simplemente **no existe en el corpus**.

Por ejemplo, una pregunta comparativa como:

> ¿Qué riesgos añadió Microsoft entre FY2024 y FY2025?

puede requerir realizar varias búsquedas y posteriormente comparar los resultados.
De forma similar, para responder a una pregunta sobre una cifra financiera exacta,
la fuente más adecuada puede no ser el texto del informe, sino directamente los
datos estructurados XBRL.

Por este motivo, en este proyecto el *retrieval* deja de constituir la arquitectura
completa del sistema y pasa a ser **una herramienta disponible para el agente**.

El sistema implementado se aproxima así al paradigma de **Agentic RAG**: el LLM
puede decidir dinámicamente cuándo recuperar información, cuándo consultar una
fuente estructurada, cuándo realizar varias búsquedas y cuándo comprobar primero
si la información solicitada está disponible.

## Arquitectura del agente

El agente dispone de cuatro herramientas principales:

| Herramienta | Función |
| --- | --- |
| `list_available()` | Consultar qué compañías, informes y periodos están disponibles en el corpus |
| `get_xbrl_fact()` | Recuperar cifras financieras estructuradas procedentes de XBRL |
| `search_filings()` | Realizar búsqueda semántica sobre el contenido de los informes 10-K |
| `read_section()` | Recuperar el contenido completo de una sección concreta de un informe |

Estas herramientas proporcionan mecanismos de recuperación con características
diferentes. En particular, `get_xbrl_fact()` permite obtener información
estructurada de forma precisa y determinista, mientras que `search_filings()`
permite localizar información no estructurada mediante similitud semántica.

La selección de la herramienta forma, por tanto, parte del propio problema de
razonamiento. El agente debe determinar no solo **qué respuesta proporcionar**,
sino también **qué fuente y qué procedimiento son los más adecuados para
obtenerla**.

## Bucle de razonamiento y uso de herramientas

Sobre estas herramientas se construye un **bucle agéntico inspirado en ReAct
(Reason + Act)**. El modelo puede alternar entre razonamiento y ejecución de
acciones, observar los resultados obtenidos y decidir el siguiente paso hasta
disponer de evidencia suficiente para generar una respuesta.

De forma simplificada, el proceso sigue el esquema:

**Pregunta → razonamiento → selección de herramienta → observación → nuevo
razonamiento → ... → respuesta final**

Este mecanismo permite resolver consultas que requieren múltiples pasos, como
comparaciones entre ejercicios, búsquedas sucesivas o combinación de información
estructurada y no estructurada.

## Objetivo de la práctica

El objetivo final es desarrollar y evaluar un agente capaz de responder preguntas
sobre informes financieros de forma **precisa, trazable y eficiente**.

Para ello se trabajará sobre un corpus de informes 10-K y se analizará no solo la
calidad de la respuesta final, sino también la **trayectoria seguida por el
agente**: qué herramientas utiliza, en qué orden y si la fuente seleccionada es
adecuada para cada tipo de consulta.

Esta distinción resulta especialmente relevante en un contexto financiero. Una
respuesta numéricamente correcta no implica necesariamente que el procedimiento
utilizado sea adecuado: cuando existe una fuente estructurada y auditable para
una cifra financiera, recuperar esa cifra directamente resulta preferible a
inferirla a partir de fragmentos de texto.

El proyecto busca, por tanto, combinar las capacidades de razonamiento de los
LLMs con mecanismos de recuperación especializados, manteniendo como principios
centrales la **precisión, la trazabilidad y la selección adecuada de fuentes**.

# 1. Instalación de liberías

Versiones de las librerías fijadas porque langchain las actualiza constantemente.

In [1]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-openrouter==0.2.8 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2
print("Instalación terminada.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 51.2 MB/s eta 0:00:00
Instalación terminada.


In [2]:
%pip install -q langchain-google-genai langchain-groq langchain-cerebras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 15.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


# 2. Configuramos la API y elegimos el modelo a utilizar

Mientras trabajamos con el free tier de Gemini (5-20 peticiones/minuto o
día según el modelo, insuficiente para iterar con comodidad — y el
crédito de $300 de Google Cloud no aplica a esta API desde marzo de
2026), usamos una cascada con `.with_fallbacks()`: si Gemini fallaba por
cuota, reintentaba con Groq y, si también fallaba, con Cerebras.

Activamos el pago (Prepay) en la API de Gemini antes de generar el
baseline, así que **retiramos la cascada**: el baseline y todas las
evaluaciones posteriores usan un único modelo fijo
(`gemini-3.8-flash`), sin fallback. Es necesario para que la comparación
baseline-vs-final sea válida — con la cascada, distintas preguntas
podrían haber sido respondidas por modelos distintos, lo que invalidaría
la comparación.

In [4]:
# Conectamos la API

from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

MODELO = "google_genai:gemini-3.8-flash"

print("Gemini configurado correctamente.")

Gemini configurado correctamente.


In [5]:
# Comprobación de que la API funbciona

from langchain.chat_models import init_chat_model

llm = init_chat_model(MODELO)

respuesta = llm.invoke("Responde únicamente: conexión correcta")

print(respuesta.content)

[{'type': 'text', 'text': 'conexión correcta', 'extras': {'signature': 'EpEECo4EAWkUfRNm2H3TlSl7LYzOB5w2BuBwmEFrvHGM9kDcST7H7qUTyCKJpTHhoe7y2ViYhoeJ9osACtnODabn2LW4HaGAbT1N1TPz9+yslDV69QtSa6klJoGBMelnWdLX56G0AKc9oemowEhVzhAZ/PIU4NMDhfF0DhrX7lfQByNIqcSy8W5soXWyy+gML1qbT1iYMLh4D0a48AwMgPE2Ri4pivr4MB/NgHRrybS8FkkoCBrUJ4cHo1gLq3eUAE4bOl4fQ0BaGbC1Ep06Cv1TJd+NxOHB5pFrke1e3FbcGs7jpo9Ue26VlYpeUVbhJbIAbhh7yA76Kpy/lsDK24IiqIaK6Bc+Q5Apy/KVrypVKYyfHGXiJkAxZ9UetJXPpPc/Mc+qjcIGxPZ02PnGHWUEZ3omOEET9CDTSFIWq0mi+crCb5Ga7UTI5Q9t3IgCEKeit6YVczNsUJw5ftESvCsZazHlMj14X/KaNpLAEdAnoyQt5GDJCkmsgBonVY5VstYA3OP8cG3H61rJHVx33WRc/+s9W2cr0AfVMpF16tCjfBqvA/33ipVgpl4/fpDyTClLwvs/mT/vCEUL2wUZ57tU0GgFplSh4h44ZH43gzzibcIPDMGw5K9tINLqCNXy/A7XbFT+aDgefq9S3KcZBvPftTsNe8kL461LAnTo4zw3nWLIGmz2VNfmc6PEUl0OS6lVTQ=='}}]


# 3. Conectamos las bases de datos desde una carpeta

In [7]:
from google.colab import drive
from pathlib import Path

# Montar Google Drive
drive.mount("/content/drive")

# Carpeta compartida que contiene los archivos originales de la práctica
DATA_DIR = Path("/content/drive/MyDrive/MIAX_Taller_NLP")

# Carpeta temporal de Colab donde se descomprimirá el corpus y el índice
CORPUS_DIR = Path("/content/corpus")

assert DATA_DIR.exists(), f"No se encuentra la carpeta: {DATA_DIR}"

print("Carpeta de datos encontrada:")
for archivo in DATA_DIR.iterdir():
    print(" -", archivo.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta de datos encontrada:
 - corpus_miax_2026.zip
 - demo_traza.json
 - golden_set_ejemplo.jsonl
 - indice_faiss.zip
 - miax_s1.py


In [8]:
import hashlib
import zipfile

PAQUETES = [
    (
        "corpus_miax_2026.zip",
        "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4",
    ),
    (
        "indice_faiss.zip",
        "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655",
    ),
]


def sha256(ruta):
    """Calcula el hash SHA-256 de un archivo."""
    hash_ = hashlib.sha256()

    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            hash_.update(bloque)

    return hash_.hexdigest()


# Comprobar y descomprimir los dos paquetes
for nombre, hash_esperado in PAQUETES:

    ruta = DATA_DIR / nombre

    assert ruta.is_file(), f"No se encuentra {nombre} en {DATA_DIR}"

    hash_obtenido = sha256(ruta)

    assert hash_obtenido == hash_esperado, (
        f"{nombre} no coincide con la versión esperada.\n"
        f"Esperado: {hash_esperado}\n"
        f"Obtenido: {hash_obtenido}"
    )

    with zipfile.ZipFile(ruta) as zip_file:
        zip_file.extractall(CORPUS_DIR)


# Comprobar que el índice FAISS corresponde exactamente al corpus
hash_chunks = sha256(CORPUS_DIR / "chunks.jsonl")

for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):

    ruta_manifiesto = CORPUS_DIR / manifiesto

    if ruta_manifiesto.exists():
        assert hash_chunks in ruta_manifiesto.read_text(encoding="utf-8"), (
            f"El índice FAISS no corresponde con los chunks del corpus "
            f"({manifiesto})."
        )


print("Corpus e índice FAISS preparados correctamente.")

Corpus e índice FAISS preparados correctamente.


# 4. Análisis del 10-K. ¿Qué hay en los datos?

El 10-K es el informe anual que toda empresa cotizada en EE. UU. presenta ante
la SEC. Es un documento normalizado: los mismos epígrafes, en el mismo orden,
todos los años y en todas las compañías. Eso es lo que lo hace utilizable como
corpus.

Nos quedamos con cuatro epígrafes, que son donde está lo que se puede
preguntar:

| Item | Qué contiene | Qué se le pregunta |
| --- | --- | --- |
| **1A** · Risk Factors | Los riesgos que la compañía declara | Qué riesgos nuevos aparecen, cómo cambian entre ejercicios |
| **7** · MD&A | La dirección explicando sus propios resultados | Por qué subió o bajó una magnitud |
| **7A** · Market Risk | Exposición a tipos, divisa y precios | Cuantitativo y corto |
| **8** · Financial Statements | Los estados financieros y sus notas | Cifras, y de dónde salen |

Dos cosas que hay que saber del corpus antes de tocarlo:

**`fiscal_year` no es el año de presentación.** Las seis compañías cierran
ejercicio en cuatro meses distintos —NVDA en enero, MSFT en junio, AAPL en
septiembre, y GOOGL, META y AMZN en diciembre—, y está elegido así a
propósito. El 10-K de NVDA FY2025 se presentó en febrero de 2025; el de
Alphabet FY2025, en febrero de **2026**. Quien razone por fecha de
presentación se equivoca.

**NVIDIA no pone sus estados financieros bajo el Item 8.** Los deja bajo el
Item 15 y en el 8 escribe una remisión de dos líneas. El corpus sirve el
contenido correcto bajo la clave `"8"` y deja constancia en el campo
`item_origen`. Si no lo hiciera, `read_section("NVDA", 2025, "8")` devolvería
cuarenta tokens inútiles.

In [9]:
# Cuánto ocupa un 10-K. Los tokens vienen precalculados en el corpus: contar
# en vivo tardaría más que la clase entera.
import json
import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open("corpus/secciones.jsonl", encoding="utf-8")
)

tabla = secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {secciones.n_tokens.sum():,} tokens en "
      f"{len(secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
print(f"Informe mayor: {tabla['TOTAL'].max():,} · menor: "
      f"{tabla['TOTAL'].min():,}")

mayor = secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la
# columna. Con una columna que se llama 'item' hay que usar corchetes.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")

item                   1A      7    7A      8  TOTAL
ticker fiscal_year                                  
AAPL   2024         11663   3814   612  15999  32088
       2025         11626   4294   612  16358  32890
AMZN   2024         10318   9597  1614  28097  49626
       2025         10516   9034  1547  29103  50200
GOOGL  2024         14727  11947  1877  30380  58931
       2025         14984  10648  1579  31845  59056
META   2024         33573  12621  1128  28501  75823
       2025         34751  12518  1144  32351  80764
MSFT   2024         12650  10295   409  28455  51809
       2025         11793   9510   409  26500  48212
NVDA   2024         18681   8348   635  26909  54573
       2025         19476   7824   638  27209  55147

Corpus entero: 649,119 tokens en 48 secciones
Informe medio: 54,093 tokens
Informe mayor: 80,764 · menor: 32,088
Sección mayor: META FY2025 Item 1A con 34,751 tokens


Vamos a analizar un poco qué hay en cada archivo.

In [1]:
print("Contenido de /content/corpus:\n")

for archivo in sorted(CORPUS_DIR.rglob("*")):
    if archivo.is_file():
        tamaño_mb = archivo.stat().st_size / 1e6
        print(f"{archivo.relative_to(CORPUS_DIR)}  →  {tamaño_mb:.2f} MB")

Contenido de /content/corpus:



NameError: name 'CORPUS_DIR' is not defined